# <Project Name>

| Field | Value |
|-------|-------|
| **Authors** | Kayla Hooper & Diya Gumaste |
| **Course** | ADS 5834 — Data Science in Practice |
| **Notebook** | modeling |
| **Created Date** | 2026-11-19 |
| **Last Updated Date** | 2026-11-19 |
| **Last Updated By** | Kayla Hooper |
| **Technical Owner Email** | khooper5@vt.edu |
| **Business Owner Email** | khooper5@vt.edu |

## Project Description


## Notebook Description

For each target, we compare logistic regression, shallow decision tree, and gradient boosting. Hyper-parameters are tuned with 5-fold CV on train(PR-AUC); the validation set picks the champion; test is untouched until notebook 05. Everything is tracked in MLflow and the champions are registered in Unity Catalog with alias champion.

In [0]:
SRC_TABLE = "workspace.insurance_fraud.auto_insurance_fraud"   # READ-ONLY warehouse table
DM = "workspace.dm_final_project"  # ALL derived work goes here
SEED = 42
assert "insurance_fraud" not in DM, "Never write to the warehouse schema"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, mlflow, mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

mlflow.set_registry_uri("databricks-uc")
user = spark.sql("select current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/insurance_fraud_models")

feat = spark.table(f"{DM}.features_all").toPandas()
FEATURES = spark.table(f"{DM}.feature_catalog").toPandas()["feature"].tolist()
TARGETS = {"organized": "is_organized_fraud", "opportunistic": "is_opportunistic_fraud"}
tr, va = feat[feat.split == "train"], feat[feat.split == "val"]
print(len(tr), len(va), len(FEATURES), "features")

## 1. Candidate models and search grids

In [0]:
def candidates():
    imp = lambda: ("imp", SimpleImputer(strategy="median"))
    return {
        "logreg_l1": (Pipeline([imp(), ("sc", StandardScaler()),
                       ("clf", LogisticRegression(penalty="l1", solver="liblinear", class_weight="balanced", max_iter=2000, random_state=SEED))]),
                      {"clf__C": [0.01, 0.03, 0.1, 0.3, 1.0]}),
        "decision_tree": (Pipeline([imp(), ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=SEED))]),
                          {"clf__max_depth": [3, 4, 5, 6], "clf__min_samples_leaf": [50, 100, 200]}),
        "hist_gb_challenger": (Pipeline([("clf", HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED))]),
                               {"clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1], "clf__max_iter": [200]}),
    }
INTERPRETABLE = {"logreg_l1", "decision_tree"}

def best_f1(y, p):
    pr, rc, _ = precision_recall_curve(y, p)
    f1 = 2 * pr * rc / np.clip(pr + rc, 1e-9, None)
    return float(np.nanmax(f1))

## 2. Train, tune, track

In [0]:
results, fitted, run_ids = [], {}, {}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

for tkey, tcol in TARGETS.items():
    Xtr, ytr = tr[FEATURES].astype(float), tr[tcol].astype(int)
    Xva, yva = va[FEATURES].astype(float), va[tcol].astype(int)
    prevalence = ytr.mean()
    with mlflow.start_run(run_name=f"{tkey}_search") as parent:
        mlflow.log_param("target", tcol); mlflow.log_param("n_features", len(FEATURES)); mlflow.log_metric("train_prevalence", prevalence)
        for name, (pipe, grid) in candidates().items():
            with mlflow.start_run(run_name=f"{tkey}_{name}", nested=True) as run:
                gs = GridSearchCV(pipe, grid, scoring="average_precision", cv=cv, n_jobs=-1, refit=True)
                gs.fit(Xtr, ytr)
                p_va = gs.predict_proba(Xva)[:, 1]
                m = dict(cv_pr_auc=gs.best_score_, val_pr_auc=average_precision_score(yva, p_va), val_roc_auc=roc_auc_score(yva, p_va),
                         val_best_f1=best_f1(yva, p_va), val_prevalence=yva.mean())
                mlflow.log_params({k.replace("clf__", ""): v for k, v in gs.best_params_.items()}); mlflow.log_metrics(m)
                mlflow.set_tags({"target": tkey, "model": name, "interpretable": name in INTERPRETABLE})
                sig = infer_signature(Xtr.head(50), gs.predict_proba(Xtr.head(50))[:, 1])
                mlflow.sklearn.log_model(gs.best_estimator_, "model", signature=sig, input_example=Xtr.head(5))
                fitted[(tkey, name)] = gs.best_estimator_; run_ids[(tkey, name)] = run.info.run_id
                results.append(dict(target=tkey, model=name, interpretable=name in INTERPRETABLE, best_params=str(gs.best_params_), **m))
res = pd.DataFrame(results)
display(res.round(4))

## 3. Champion selection
Rule: best **interpretable** model by validation PR-AUC (regulators require an explainable EUO rationale). The challenger gap is reported so the business can see the accuracy/explainability trade-off.

In [0]:
champions = {}
for tkey in TARGETS:
    r = res[res.target == tkey]
    champ = r[r.interpretable].sort_values("val_pr_auc", ascending=False).iloc[0]
    chall = r[r.model == "hist_gb_challenger"].iloc[0]
    champions[tkey] = champ["model"]
    gap = chall.val_pr_auc - champ.val_pr_auc
    print(f"{tkey:14s} champion={champ.model:14s} val PR-AUC={champ.val_pr_auc:.3f} | challenger={chall.val_pr_auc:.3f} (gap {gap:+.3f})"
          + ("  <-- gap is material, discuss with compliance" if gap > 0.03 else ""))

In [0]:
registered = {}
for tkey, mname in champions.items():
    uc_name = f"{DM}.fraud_{tkey}_model"
    try:
        mv = mlflow.register_model(f"runs:/{run_ids[(tkey, mname)]}/model", uc_name)
        mlflow.MlflowClient().set_registered_model_alias(uc_name, "champion", mv.version)
        registered[tkey] = (uc_name, mv.version); print("registered", uc_name, "v", mv.version, "alias=champion")
    except Exception as e:
        print("Registration failed for", uc_name, "->", e)

res["is_champion"] = [champions.get(t) == m for t, m in zip(res.target, res.model)]
spark.createDataFrame(res.astype({"best_params": str})).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.model_comparison")
pd.DataFrame({"feature": FEATURES}).pipe(lambda d: spark.createDataFrame(d)).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.model_feature_list")

## 4. Interpret the champions (this is the regulator-facing explanation)

In [0]:
for tkey, mname in champions.items():
    pipe = fitted[(tkey, mname)]
    print("=" * 90); print(f"{tkey.upper()} - {mname}")
    if mname == "logreg_l1":
        coef = pd.Series(pipe.named_steps["clf"].coef_[0], index=FEATURES)
        nz = coef[coef != 0].sort_values()
        print(f"{len(nz)} of {len(FEATURES)} features kept by L1 (standardized coefficients; + raises fraud score)")
        top = pd.concat([nz.head(8), nz.tail(12)]).drop_duplicates()
        fig, ax = plt.subplots(figsize=(8, 0.35 * len(top) + 1)); top.plot.barh(ax=ax, color=["#c0392b" if v > 0 else "#2e86c1" for v in top]); ax.set_title(f"{tkey}: logistic coefficients"); plt.tight_layout(); plt.show()
        display(nz.sort_values(key=abs, ascending=False).rename("coef").reset_index().rename(columns={"index": "feature"}))
    else:
        print(export_text(pipe.named_steps["clf"], feature_names=FEATURES, max_depth=4))
        imp = pd.Series(pipe.named_steps["clf"].feature_importances_, index=FEATURES).sort_values(ascending=False).head(15)
        display(imp.rename("importance").reset_index().rename(columns={"index": "feature"}))

In [0]:
# Challenger permutation importance on validation (what is the black box using?)
from sklearn.inspection import permutation_importance
for tkey, tcol in TARGETS.items():
    pipe = fitted[(tkey, "hist_gb_challenger")]
    pi = permutation_importance(pipe, va[FEATURES].astype(float), va[tcol], scoring="average_precision", n_repeats=5, random_state=SEED, n_jobs=-1)
    s = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).head(12)
    print(f"\n{tkey}: top challenger features (permutation, PR-AUC drop)"); display(s.round(4).rename("importance").reset_index())